<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/_banners/opim5641_banner.svg" width="100%" alt="OPIM 5641 banner"/>

**Dr. Dave Wanik - Operations and Information Management - University of Connecticut**

---

# Assignment 5 — Regression as Optimization: Baseball Heights and Weights

In Studio 4.1 you fit a regression to ten pharmacies by writing it as an optimization model. Now you do it
for **1,033 Major League Baseball players**: how much does a player weigh, given his height?

**You do not write any model code.** The model from Studio 4.1 is already here, wrapped up so that a whole
scenario is **one line**. You work through the steps top to bottom: run a cell, read what it prints, and fill
in the blanks.

| Step | What you do | Points |
|---|---|---|
| 1 | Load the players (just run it) | - |
| 2 | Load the model (just run it) | - |
| 3 | The base model: a straight line, squared error | 20 |
| 4 | Change the error: absolute error | 20 |
| 5 | Change the form: a power curve and a quadratic | 30 |
| 6 | Add a business rule: never under-predict | 15 |
| 7 | Your recommendation | 15 |

### The model you are fitting

$$
\begin{aligned}
\min \quad & \sum_{i=1}^{n} \text{error}(y_i - z_i) && \text{(squared or absolute)} \\
\text{subject to} \quad & y_i = f(x_i) \quad \text{for every player } i && \text{(the form: } a + b\,x,\; a\,x^b \text{ or } a + b\,x + c\,x^2) \\
\end{aligned}
$$

where $x_i$ is player $i$'s height in inches, $z_i$ is his actual weight in pounds, and $y_i$ is the model's
placeholder prediction, the same trick as Studio 4.1.

## Import modules (download Pyomo and the ipopt solver)

Run this first, every time. It installs Pyomo and **ipopt**. It takes about a minute.

In [ ]:
# install Pyomo and the ipopt solver in Colab (idaes-pse brings both)
import sys
import os

if 'google.colab' in sys.modules:
    !pip install -q idaes-pse --pre
    !idaes get-extensions --to ./bin
    os.environ['PATH'] += ':bin'

%matplotlib inline
from pylab import *
import numpy as np
import pandas as pd
from pyomo.environ import *

# where the setup cell put the solver
ipopt_path = '/content/bin/ipopt'

# Step 1 · Load the players

**Just run it.** The file is the classic MLB heights-and-weights data set: name, team, position, height
(inches), weight (pounds) and age for 1,034 players.

One player is missing his weight, so the cell shows you that row and drops it. Real data always has a few
holes like this; the model cannot use a row without a weight.

In [ ]:
players = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/Assignments/data/mlb_height_weight.csv', index_col=0)
print(len(players), "players in the file")

# the row with no weight
missing = players[players['Weight(pounds)'].isna()]
print("missing a weight:")
print(missing)

players = players.dropna(subset=['Weight(pounds)'])
players = players.reset_index(drop=True)
print()
print(len(players), "players after dropping it")

# X = height (the input), Z = weight (the actual value we want to predict)
X = []
Z = []
for i in range(len(players)):
    X.append(float(players.loc[i, 'Height(inches)']))
    Z.append(float(players.loc[i, 'Weight(pounds)']))
n_points = len(X)

players.head()

In [ ]:
# look before you model
scatter(X, Z, alpha=0.3, color='black')
xlabel('Height (inches)')
ylabel('Weight (pounds)')
title('1,033 MLB players')
show()

# Step 2 · Load the model

**Just run this cell.** The code is hidden because you do not need to change it (double-click the title to
see it). It is the Studio 4.1 model: one placeholder $y_i$ per player, one constraint per player that forces
$y_i$ to follow the form of the model, and the error you choose in the objective.

After this cell, a whole scenario is one line:

`fit_model("a name", model_form="linear", error_type="squared")`

It prints the fitted equation, the total squared error (one yardstick for every scenario) and predictions for
three heights, and it adds the scenario to a scoreboard.

In [ ]:
#@title Step 2 · Load the model (just run this cell - the code is hidden on purpose) { display-mode: "form" }
import logging
logging.getLogger('pyomo.core').setLevel(logging.ERROR)
from IPython.display import display, Math

scoreboard = {}
curves = {}
grid = np.linspace(66, 84, 200)
check_heights = [70, 74, 78]        # 5'10", 6'2" and 6'6"


def fit_model(scenario, model_form="linear", error_type="squared", never_underpredict=False):
    model = ConcreteModel()

    # decision variables: the parameters, then one placeholder prediction per player
    model.a = Var(domain=Reals, initialize=0)
    model.b = Var(domain=Reals, initialize=1)
    if model_form == "power":
        model.a = 1.0
        model.b = 1.2
    if model_form == "quadratic":
        model.c = Var(domain=Reals, initialize=0)
    model.y = Var(range(n_points), domain=Reals, initialize=200)

    # one constraint per player: the form of the model
    model.constraints = ConstraintList()
    for i in range(n_points):
        x = X[i]
        if model_form == "linear":
            model.constraints.add(model.y[i] == model.a + model.b * x)
        if model_form == "power":
            model.constraints.add(model.y[i] == model.a * x ** model.b)
        if model_form == "quadratic":
            model.constraints.add(model.y[i] == model.a + model.b * x + model.c * x ** 2)

    # the objective: the error you choose
    obj_expr = 0
    if error_type == "squared":
        for i in range(n_points):
            obj_expr += (model.y[i] - Z[i]) ** 2
    if error_type == "absolute":
        model.d = Var(range(n_points), domain=NonNegativeReals)      # d[i] = |y[i] - Z[i]|
        for i in range(n_points):
            model.constraints.add(model.d[i] >= model.y[i] - Z[i])
            model.constraints.add(model.d[i] >= -(model.y[i] - Z[i]))
            obj_expr += model.d[i]
    model.error = Objective(expr=obj_expr, sense=minimize)

    # a business rule: never predict less than a player actually weighs
    if never_underpredict:
        for i in range(n_points):
            model.constraints.add(model.y[i] >= Z[i])

    results = SolverFactory('ipopt', executable=ipopt_path).solve(model)

    a = model.a()
    b = model.b()
    c = 0
    if model_form == "quadratic":
        c = model.c()
    if model_form == "linear":
        fitted = r"\hat{y} = %.2f %+.4f\,x" % (a, b)
    if model_form == "power":
        fitted = r"\hat{y} = %.4f\,x^{%.4f}" % (a, b)
    if model_form == "quadratic":
        fitted = r"\hat{y} = %.2f %+.4f\,x %+.6f\,x^2" % (a, b, c)

    def predict(h):
        if model_form == "linear":
            return a + b * h
        if model_form == "power":
            return a * h ** b
        return a + b * h + c * h ** 2

    sse = 0
    for i in range(n_points):
        sse += (model.y[i]() - Z[i]) ** 2

    print("SCENARIO:", scenario, "   (solver:", results.solver.termination_condition, ")")
    display(Math(fitted))
    print("total squared error = {:,.0f}".format(sse))
    row = {'form': model_form, 'error': error_type, 'rule': never_underpredict, 'a': round(a, 4), 'b': round(b, 4),
           'c': round(c, 6), 'squared error': round(sse)}
    for h in check_heights:
        print("predicted weight at %d inches: %.1f lb" % (h, predict(h)))
        row['weight at %d in' % h] = round(predict(h), 1)
    scoreboard[scenario] = row
    curve = []
    for h in grid:
        curve.append(predict(h))
    curves[scenario] = curve


def check_with_sklearn():
    from sklearn.linear_model import LinearRegression
    ols = LinearRegression()
    ols.fit(np.array(X).reshape(-1, 1), Z)
    print("sklearn:  a = %.2f   b = %.4f" % (ols.intercept_, ols.coef_[0]))


def show_all():
    display(pd.DataFrame(scoreboard).T)
    figure(figsize=(10, 6))
    scatter(X, Z, alpha=0.2, color='black', label='players')
    for name in curves:
        plot(grid, curves[name], linewidth=2.5, label=name)
    xlabel('Height (inches)')
    ylabel('Weight (pounds)')
    title('Same players, different choices')
    legend(fontsize=8)
    show()


print("the model is loaded")

# Step 3 · The base model: a straight line, squared error (20 points)

$$\min \sum_i (y_i - z_i)^2 \quad \text{subject to} \quad y_i = a + b\,x_i$$

Run both cells. The second one fits the same line with `sklearn`, so you can check the optimization model.

In [ ]:
fit_model("1 linear, squared", model_form="linear", error_type="squared")

In [ ]:
check_with_sklearn()

### YOUR ANSWER - Step 3

Double-click this cell and fill in the blanks.

**3a (5 points).** The fitted line is weight = \_\_\_\_\_\_ + \_\_\_\_\_\_ × height. Does it match `sklearn`? \_\_\_\_ (yes or no)

**3b (5 points).** In plain words, the slope $b$ means: each extra inch of height adds about \_\_\_\_\_\_ pounds.

**3c (5 points).** The model predicts \_\_\_\_\_\_ lb for a 6'2" player (74 inches) and \_\_\_\_\_\_ lb for a 6'6" player (78 inches).

**3d (5 points).** The intercept $a$ is the predicted weight of a player who is 0 inches tall. Why is a negative number here not a problem? One sentence.

*your sentence*

# Step 4 · Change the error: absolute error (20 points)

Same line, a different objective:

$$\min \sum_i \lvert y_i - z_i \rvert \quad \text{subject to} \quad y_i = a + b\,x_i$$

Remember why the model uses $d_i$ with two floors instead of `abs()`: the solver follows the slope of the
objective downhill, and a V has no slope at its point. Just run it.

In [ ]:
fit_model("2 linear, absolute", model_form="linear", error_type="absolute")

### YOUR ANSWER - Step 4

**4a (5 points).** With absolute error the line is weight = \_\_\_\_\_\_ + \_\_\_\_\_\_ × height.

**4b (5 points).** Its total squared error is \_\_\_\_\_\_, compared with \_\_\_\_\_\_ in Step 3. Why must the
Step 3 line have the smaller squared error? One sentence.

*your sentence*

**4c (10 points).** Squared error punishes one big miss much more than several small ones. Look at the scatter
plot in Step 1: are there players far from the cloud? Which error would you trust more for this data, and why?
Two sentences.

*your sentences*

# Step 5 · Change the form: a power curve and a quadratic (30 points)

Two more forms, both with squared error:

$$\text{power: } y_i = a\,x_i^{\,b} \qquad\qquad \text{quadratic: } y_i = a + b\,x_i + c\,x_i^2$$

Run both, then `show_all()` puts every scenario so far in one table and one picture.

In [ ]:
fit_model("3 power, squared", model_form="power", error_type="squared")

In [ ]:
fit_model("4 quadratic, squared", model_form="quadratic", error_type="squared")

In [ ]:
show_all()

### YOUR ANSWER - Step 5

**5a (10 points).** Fill in the total squared error of each scenario:
linear \_\_\_\_\_\_, power \_\_\_\_\_\_, quadratic \_\_\_\_\_\_.

**5b (10 points).** Which form has the lowest squared error, and by roughly how much (in percent) does it beat
the straight line? Is that difference worth a more complicated model? Two sentences.

*your sentences*

**5c (10 points).** In Studio 4.1, the pharmacy curves looked very different from each other. Here they almost sit
on top of each other. Why do you think the choice of form matters so much less with these players? One or two
sentences. (Hint: how many data points, and how curved does the cloud look?)

*your sentences*

# Step 6 · Add a business rule: never under-predict (15 points)

A uniform company sizes every player's jersey and pants from the model's predicted weight. An outfit that is
too big can be taken in; one that is too small is useless. So the company asks for a model that **never
predicts less than a player actually weighs**. In an optimization model, that is one more constraint per player:

$$y_i \ge z_i \qquad \text{for every player } i$$

Just run it, then look at the picture again.

In [ ]:
fit_model("5 linear, never under-predict", model_form="linear", error_type="squared", never_underpredict=True)

In [ ]:
show_all()

### YOUR ANSWER - Step 6

**6a (5 points).** With the rule, the line is weight = \_\_\_\_\_\_ + \_\_\_\_\_\_ × height, and it predicts
\_\_\_\_\_\_ lb for a 6'2" player (compared with \_\_\_\_\_\_ lb in Step 3).

**6b (10 points).** The squared error jumped from about 326,000 to about \_\_\_\_\_\_\_\_\_. Which players decide
where this line sits, and would you hand this model to the uniform company? Two or three sentences.

*your sentences*

# Step 7 · Your recommendation (15 points)

The team's strength coach has never heard of Pyomo. In **four or five sentences**, tell them:

1. which model you recommend for predicting a player's weight from his height, and why,
2. what it predicts for a 6'2" player, and
3. one thing the "never under-predict" scenario taught you about adding business rules.

### YOUR ANSWER - Step 7

*your four or five sentences*

# Turn it in

1. Click **Runtime → Run all** and wait for it to finish. Scroll down and make sure every step printed something
   and there are no red error messages.
2. Check that every **YOUR ANSWER** cell is filled in.
3. **File → Download → Download .ipynb**, and upload that file to Assignment 5 on HuskyCT.

**Due Sunday, November 8 at 11:59 PM.**